In [1]:
import pandas as pd
from pathlib import Path

# Path.cwd() gets the folder your notebook is currently in.
# .parent goes up one level to the project root.
BASE_DIR = Path.cwd().parent
labels_path = BASE_DIR / "data" / "processed" / "matrices" / "master_clinical_labels.csv"

if not labels_path.exists():
    print(f"[!] Still can't find it. Jupyter is currently running from: {Path.cwd()}")
    print(f"[!] It is trying to look here: {labels_path}")
    print("Try adding another '.parent' to BASE_DIR if your notebook is nested deeper.")
else:
    df = pd.read_csv(labels_path)

    # Calculate how many patients come from each cohort
    cohort_counts = df['Dataset'].value_counts().reset_index()
    cohort_counts.columns = ['Cohort', 'Patient_Count']

    # Print the numbers for your Excel sheet
    print(f"Total Patients: {len(df)}\n")
    print(cohort_counts.to_string(index=False))

Total Patients: 1636

   Cohort  Patient_Count
 GSE65682            479
GSE185263            345
GSE236713            324
GSE272769            161
 GSE54514            127
 GSE95233            102
 GSE26440             98


In [2]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent
labels_path = BASE_DIR / "data" / "processed" / "matrices" / "master_clinical_labels.csv"
df = pd.read_csv(labels_path)

# Map the 0/1 labels to readable text for the table
df['Outcome'] = df['Mortality'].map({0: 'Survivor', 1: 'Non-Survivor'})

# Create a clean cross-tabulation table
breakdown = pd.crosstab(
    index=df['Dataset'], 
    columns=df['Outcome'], 
    margins=True, 
    margins_name="Total"
)

# Print the matrix for Excel
print("MORTALITY BREAKDOWN PER COHORT:\n")
print(breakdown)

MORTALITY BREAKDOWN PER COHORT:

Outcome    Non-Survivor  Survivor  Total
Dataset                                 
GSE185263            52       293    345
GSE236713            59       265    324
GSE26440             17        81     98
GSE272769            60       101    161
GSE54514             31        96    127
GSE65682            114       365    479
GSE95233             34        68    102
Total               367      1269   1636


In [5]:
import pandas as pd
from pathlib import Path

# Path.cwd() is your 'notebooks' folder. .parent goes up to 'Multimodal_prediction_sepsis'
BASE_DIR = Path.cwd().parent
META_DIR = BASE_DIR / "data" / "raw" / "geo_metadata"

print(f"[*] Searching inside: {META_DIR}\n")

TARGET_COHORTS = [
    "GSE185263", "GSE236713", "GSE26440", "GSE272769",
    "GSE54514", "GSE65682", "GSE95233"
]

for cohort in TARGET_COHORTS:
    file_path = META_DIR / f"{cohort}_metadata.csv"
    
    if file_path.exists():
        df = pd.read_csv(file_path)
        
        print(f"\n" + "="*50)
        print(f"[*] {cohort} (Raw Metadata Samples: {len(df)})")
        print("="*50)
        
        # Print all columns to spot Age, Sex, Tissue, Platform
        print("AVAILABLE COLUMNS:")
        print(df.columns.tolist())
        
        # Print the first row of data to see exactly how the lab formatted it
        print("\nDATA SNEAK PEEK (Row 1):")
        print(df.iloc[0].dropna().to_dict()) 
        
    else:
        print(f"[!] ERROR: Could not find {file_path.name}")

[*] Searching inside: /workspace/data/raw/geo_metadata


[*] GSE185263 (Raw Metadata Samples: 392)
AVAILABLE COLUMNS:
['Patient_ID', 'disease state', 'age', 'sex', 'collection location', 'collection site', 'sofa 24h post admisssion', 'in hospital mortality', 'tissue']

DATA SNEAK PEEK (Row 1):
{'Patient_ID': 'GSM5608946', 'disease state': 'sepsis', 'age': 64, 'sex': 'M', 'collection location': 'colombia', 'collection site': 'Emergency Room', 'sofa 24h post admisssion': 6.0, 'in hospital mortality': 'Survived', 'tissue': 'Whole Blood'}

[*] GSE236713 (Raw Metadata Samples: 447)
AVAILABLE COLUMNS:
['Patient_ID', 'sample type', 'patient id', 'gender', 'disease', 'disease 2', 'died/survived']

DATA SNEAK PEEK (Row 1):
{'Patient_ID': 'GSM7573957', 'sample type': 'Heparinised whole  blood - purified peripheral blood leukocytes (PBLs)', 'patient id': '7', 'gender': 'F', 'disease': 'Sepsis', 'disease 2': 'Pulmonary', 'died/survived': 'Survived'}

[*] GSE26440 (Raw Metadata Samples: 130)
AVAILA

In [6]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Set up paths
BASE_DIR = Path.cwd().parent
META_DIR = BASE_DIR / "data" / "raw" / "geo_metadata"
LABELS_PATH = BASE_DIR / "data" / "processed" / "matrices" / "master_clinical_labels.csv"

# Load the master mortality labels so we only count valid patients
df_labels = pd.read_csv(LABELS_PATH)

# Map the exact column names we just discovered
COHORT_MAPPING = {
    'GSE185263': {'age': 'age', 'sex': 'sex'},
    'GSE236713': {'age': None, 'sex': 'gender'},      # Age not reported
    'GSE26440':  {'age': 'age (years)', 'sex': None}, # Sex not reported
    'GSE272769': {'age': 'age', 'sex': 'sex'},
    'GSE54514':  {'age': 'age (years)', 'sex': 'gender'},
    'GSE65682':  {'age': 'age', 'sex': 'gender'},
    'GSE95233':  {'age': 'age', 'sex': 'gender'}
}

# Helper functions to clean the messy strings (e.g., "M", "Male", "64 yrs")
def extract_age(val):
    if pd.isna(val): return np.nan
    match = re.search(r'\d+(\.\d+)?', str(val))
    return float(match.group()) if match else np.nan

def clean_sex(val):
    if pd.isna(val): return "Unknown"
    v = str(val).lower().strip()
    if v in ['m', 'male']: return 'Male'
    if v in ['f', 'female']: return 'Female'
    return "Unknown"

print("--- DEMOGRAPHICS FOR EXCEL TABLE 1 ---\n")

for cohort, mapping in COHORT_MAPPING.items():
    # 1. Get ONLY the valid patients for this specific cohort
    valid_patients = df_labels[df_labels['Dataset'] == cohort]['Patient_ID'].tolist()
    
    # 2. Load the raw metadata
    meta_path = META_DIR / f"{cohort}_metadata.csv"
    df_meta = pd.read_csv(meta_path)
    
    # 3. Filter metadata down to just our valid patients
    df_meta = df_meta[df_meta['Patient_ID'].isin(valid_patients)]
    
    # 4. Calculate Age (Median [Q1-Q3])
    if mapping['age'] and mapping['age'] in df_meta.columns:
        ages = df_meta[mapping['age']].apply(extract_age).dropna()
        if len(ages) > 0:
            age_str = f"{ages.median():.1f} [{ages.quantile(0.25):.1f}-{ages.quantile(0.75):.1f}]"
        else:
            age_str = "All values missing"
    else:
        age_str = "Not Reported"
        
    # 5. Calculate Sex (Male / Female counts)
    if mapping['sex'] and mapping['sex'] in df_meta.columns:
        sexes = df_meta[mapping['sex']].apply(clean_sex)
        male_c = (sexes == 'Male').sum()
        female_c = (sexes == 'Female').sum()
        sex_str = f"M: {male_c}, F: {female_c}"
    else:
        sex_str = "Not Reported"
        
    # 6. Print the results for Excel
    print(f"{cohort}:")
    print(f"  Total Patients : {len(valid_patients)}")
    print(f"  Age (Med [IQR]): {age_str}")
    print(f"  Sex (Count)    : {sex_str}\n")

--- DEMOGRAPHICS FOR EXCEL TABLE 1 ---

GSE185263:
  Total Patients : 345
  Age (Med [IQR]): 61.0 [44.0-73.0]
  Sex (Count)    : M: 201, F: 144

GSE236713:
  Total Patients : 324
  Age (Med [IQR]): Not Reported
  Sex (Count)    : M: 167, F: 157

GSE26440:
  Total Patients : 98
  Age (Med [IQR]): 2.2 [1.0-5.9]
  Sex (Count)    : Not Reported

GSE272769:
  Total Patients : 161
  Age (Med [IQR]): 63.0 [54.0-73.0]
  Sex (Count)    : M: 74, F: 87

GSE54514:
  Total Patients : 127
  Age (Med [IQR]): 60.0 [51.0-71.0]
  Sex (Count)    : M: 52, F: 75

GSE65682:
  Total Patients : 479
  Age (Med [IQR]): 63.0 [53.0-71.0]
  Sex (Count)    : M: 272, F: 207

GSE95233:
  Total Patients : 102
  Age (Med [IQR]): 66.0 [53.2-74.2]
  Sex (Count)    : M: 62, F: 36



In [7]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# 1. Paths
BASE_DIR = Path.cwd().parent
META_DIR = BASE_DIR / "data" / "raw" / "geo_metadata"
LABELS_PATH = BASE_DIR / "data" / "processed" / "matrices" / "master_clinical_labels.csv"

df_labels = pd.read_csv(LABELS_PATH)
df_labels['Outcome'] = df_labels['Mortality'].map({0: 'Survivor', 1: 'Non-Survivor'})

# 2. Known column mappings from our previous check
COHORT_MAPPING = {
    'GSE185263': {'age': 'age', 'sex': 'sex'},
    'GSE236713': {'age': None, 'sex': 'gender'},      
    'GSE26440':  {'age': 'age (years)', 'sex': None}, 
    'GSE272769': {'age': 'age', 'sex': 'sex'},
    'GSE54514':  {'age': 'age (years)', 'sex': 'gender'},
    'GSE65682':  {'age': 'age', 'sex': 'gender'},
    'GSE95233':  {'age': 'age', 'sex': 'gender'}
}

# 3. Cleaners
def extract_age(val):
    if pd.isna(val): return np.nan
    match = re.search(r'\d+(\.\d+)?', str(val))
    return float(match.group()) if match else np.nan

def clean_sex(val):
    if pd.isna(val): return "Unknown"
    v = str(val).lower().strip()
    if v in ['m', 'male']: return 'Male'
    if v in ['f', 'female']: return 'Female'
    return "Unknown" # This safely catches the French "H"

# 4. Pool all the data
pooled_data = []

for cohort, mapping in COHORT_MAPPING.items():
    valid_patients = df_labels[df_labels['Dataset'] == cohort][['Patient_ID', 'Outcome']].copy()
    
    meta_path = META_DIR / f"{cohort}_metadata.csv"
    df_meta = pd.read_csv(meta_path)
    
    # Merge the outcome labels with the raw metadata
    df_merged = pd.merge(valid_patients, df_meta, on='Patient_ID', how='inner')
    
    # Extract age and sex safely
    if mapping['age'] and mapping['age'] in df_merged.columns:
        df_merged['Clean_Age'] = df_merged[mapping['age']].apply(extract_age)
    else:
        df_merged['Clean_Age'] = np.nan
        
    if mapping['sex'] and mapping['sex'] in df_merged.columns:
        df_merged['Clean_Sex'] = df_merged[mapping['sex']].apply(clean_sex)
    else:
        df_merged['Clean_Sex'] = "Unknown"
        
    pooled_data.append(df_merged[['Patient_ID', 'Outcome', 'Clean_Age', 'Clean_Sex']])

# 5. The Master Pooled Dataframe
master_pool = pd.concat(pooled_data, ignore_index=True)

# 6. Calculate Stats for the Printout
def get_stats(df_subset, name):
    n = len(df_subset)
    # Age calc (ignores NaNs automatically)
    ages = df_subset['Clean_Age'].dropna()
    age_str = f"{ages.median():.1f} [{ages.quantile(0.25):.1f}-{ages.quantile(0.75):.1f}]" if len(ages) > 0 else "N/A"
    age_missing = df_subset['Clean_Age'].isna().sum()
    
    # Sex calc
    males = (df_subset['Clean_Sex'] == 'Male').sum()
    females = (df_subset['Clean_Sex'] == 'Female').sum()
    unknowns = (df_subset['Clean_Sex'] == 'Unknown').sum()
    
    print(f"\n{name} (N = {n})")
    print("-" * 30)
    print(f"Age (Median [IQR]) : {age_str}  *(Missing: {age_missing})*")
    print(f"Sex - Male (%)     : {males} ({(males/n)*100:.1f}%)")
    print(f"Sex - Female (%)   : {females} ({(females/n)*100:.1f}%)")
    print(f"Sex - Unknown/NR   : {unknowns}")

print("=== MAIN MANUSCRIPT TABLE 1: POOLED COHORT DEMOGRAPHICS ===")
get_stats(master_pool, "TOTAL COHORT")
get_stats(master_pool[master_pool['Outcome'] == 'Survivor'], "SURVIVORS")
get_stats(master_pool[master_pool['Outcome'] == 'Non-Survivor'], "NON-SURVIVORS")

=== MAIN MANUSCRIPT TABLE 1: POOLED COHORT DEMOGRAPHICS ===

TOTAL COHORT (N = 1636)
------------------------------
Age (Median [IQR]) : 61.0 [47.0-71.0]  *(Missing: 324)*
Sex - Male (%)     : 828 (50.6%)
Sex - Female (%)   : 706 (43.2%)
Sex - Unknown/NR   : 102

SURVIVORS (N = 1269)
------------------------------
Age (Median [IQR]) : 60.0 [42.8-70.0]  *(Missing: 265)*
Sex - Male (%)     : 649 (51.1%)
Sex - Female (%)   : 539 (42.5%)
Sex - Unknown/NR   : 81

NON-SURVIVORS (N = 367)
------------------------------
Age (Median [IQR]) : 65.0 [53.0-74.0]  *(Missing: 59)*
Sex - Male (%)     : 179 (48.8%)
Sex - Female (%)   : 167 (45.5%)
Sex - Unknown/NR   : 21
